# Screen Comparison Summary Example

This notebook is a researcher-facing exploratory example for FungMod's guarded comparison-summary and report-output workflow. It is not an empirical validation, calibration, or literature-comparison notebook. The example uses existing registry-backed virtual-experiment records, writes standard outputs and reports, then inspects `comparison_summary.csv` guardrails before interpreting side-by-side rows.

In [ ]:
import csv
import os
from pathlib import Path

from fungal_model import environment_grid, virtual_experiment

OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", "outputs/notebooks"))
OUTPUT_DIR = OUTPUT_ROOT / "13_screen_comparison_summary_example"
REGISTRY = Path("data_registry/registry_index.yml")


Create a small public-API virtual experiment with two runtime environment-grid cases. These pH, temperature, and oxygen values are metadata-only here unless explicit response laws or condition-specific parameter records are active, so the comparison summary must not rank them or treat them as environmental response curves.

In [ ]:
study = virtual_experiment(
    fungi="beta-glucosidase source",
    substrates="cellobiose substrate",
    environments=environment_grid(temperature_C=[30.0, 35.0], ph=[5.0], oxygen="aerobic"),
    registry=REGISTRY,
)

[(report.status, report.required_processes) for report in study.preflight(mode="exploratory")]


Run one exploratory sample per case and write the standard output bundle. `quicklook=False` keeps the smoke path fast; the standard CSV tables remain the primary analysis artifacts.

In [ ]:
result = study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=13,
    output_dir=OUTPUT_DIR,
    quicklook=False,
)

result.write_summary()
result.write_manifest()
report_path = result.write_report(OUTPUT_DIR / "report", include_html=True, include_index=True)

report_path, sorted(path.name for path in OUTPUT_DIR.glob("*.csv"))[:6]


Load `comparison_summary.csv` through the public result accessor. The rows are derived from existing final-metric and threshold-time tables; they are not empirical observations and do not add calibration or validation evidence.

In [ ]:
comparison_rows = result.comparison_summary()
comparison_path = OUTPUT_DIR / "comparison_summary.csv"

assert comparison_path.exists()
assert comparison_rows
assert {"comparison_allowed", "ranking_allowed", "ranking_blocking_reason", "recommended_next_action"}.issubset(comparison_rows[0])

[(row["source_table"], row["source_metric"], row["units"]) for row in comparison_rows[:4]]


Inspect the guardrail columns before any side-by-side screening. For this metadata-only environment grid, FungMod exposes comparable source rows for inspection but blocks ranking and environmental-response plotting.

In [ ]:
guardrail_rows = [
    {
        "environment_name": row["environment_name"],
        "source_metric": row["source_metric"],
        "comparison_allowed": row["comparison_allowed"],
        "ranking_allowed": row["ranking_allowed"],
        "ranking_blocking_reason": row["ranking_blocking_reason"],
        "recommended_next_action": row["recommended_next_action"],
    }
    for row in comparison_rows
    if row["source_table"] == "final_metrics"
]

assert {row["comparison_allowed"] for row in guardrail_rows} == {"false"}
assert {row["ranking_allowed"] for row in guardrail_rows} == {"false"}
assert all("cannot be ranked" in row["ranking_blocking_reason"] for row in guardrail_rows)
assert all(row["recommended_next_action"] == "inspect_source_rows_only_do_not_rank_or_plot_as_response" for row in guardrail_rows)

guardrail_rows[:4]


The report folder is a presentation layer over the same output bundle. The Markdown report, optional HTML sidecar, and folder index help browse artifacts; they do not reinterpret scientific values.

In [ ]:
report_dir = OUTPUT_DIR / "report"
assert (report_dir / "virtual_experiment_report.md").exists()
assert (report_dir / "virtual_experiment_report.html").exists()
assert (report_dir / "index.html").exists()

index_text = (report_dir / "index.html").read_text(encoding="utf-8")
assert "comparison_summary.csv" in index_text

with comparison_path.open(newline="", encoding="utf-8") as handle:
    csv_columns = next(csv.DictReader(handle)).keys()

sorted(csv_columns)
